# DemandCast cross-paper comparison (CB-24)

Ranks every model saved by the seven paper notebooks on **one** day-ahead task, **one** test set and **one** set of metrics.

**Inputs to add in Kaggle (Add Input):**
1. the `cb-benchmark` dataset (contains `cb_benchmark.py`)
2. the output of each paper notebook after it was re-run with the CB edits (each writes `cb_predictions/`)
3. the preprocessed DemandCast data (`preprocessed_hourly`)

Primary metric: **Country_MAPE** = MAPE computed per country, then averaged (every country weighs the same). Secondary: pooled WAPE, RMSE, MAE, bias.

In [ ]:
import sys, glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

for p in glob.glob("/kaggle/input/**/cb_benchmark.py", recursive=True)[:1]:
    sys.path.insert(0, os.path.dirname(p))
from cb_benchmark import *

PRED_DIRS = sorted({os.path.dirname(f) for f in glob.glob("/kaggle/input/**/cb_predictions/*.parquet", recursive=True)})
RAW_PATH = (glob.glob("/kaggle/input/**/preprocessed_hourly*.parquet", recursive=True) +
            glob.glob("/kaggle/input/**/preprocessed_hourly*.csv", recursive=True))[0]
MIN_COVERAGE = 0.5      # a run that predicts < 50 % of the union of origins is reported but left out
PRIMARY = "Country_MAPE"
print("Prediction folders:", *PRED_DIRS, sep="\n  ")
print("Raw data:", RAW_PATH)

In [ ]:
raw = pd.read_parquet(RAW_PATH) if RAW_PATH.endswith(".parquet") else pd.read_csv(RAW_PATH, parse_dates=[CB["TIME"]])
raw[CB["TIME"]] = pd.to_datetime(raw[CB["TIME"]])

pred = cb_attach_actuals(cb_load_predictions(PRED_DIRS), raw)
FAMILY = [("REF-", "reference"), ("ARIMA", "statistical"), ("SES", "statistical"), ("Prophet", "statistical"),
          ("XGB", "tree"), ("Stacking", "ensemble"), ("TCN", "convolutional"), ("CNN", "convolutional"),
          ("VCAG", "attention"), ("Att", "attention"), ("Transformer", "attention"), ("MLP", "feedforward"),
          ("BP", "feedforward"), ("LSTM", "recurrent"), ("GRU", "recurrent")]
def family_of(model):
    return next((f for k, f in FAMILY if k.lower() in model.lower()), "other")
blank = pred["family"].fillna("").eq("")
pred.loc[blank, "family"] = pred.loc[blank, "model"].map(family_of)
keys = pred[["country", "origin"]].drop_duplicates()
base = cb_attach_actuals(pd.concat([
    cb_seasonal_naive(keys, raw, 24, "Seasonal naive 24h"),
    cb_seasonal_naive(keys, raw, 168, "Seasonal naive 168h"),
]), raw)
base = base[base["y_pred"].notna()]
allp = pd.concat([pred, base], ignore_index=True)

# sanity: every origin must be a local-23 test origin
tab = cb_origin_table(raw)
lab = cb_label(tab, keys["country"], keys["origin"])
bad = (~lab["cb_origin"]) | (lab["cb_split"] != "test")
assert not bad.any(), f"{bad.sum()} saved origins are not CB test origins - check the notebook edits"
print(allp.groupby(["notebook", "model", "feature_set", "track"])["seed"].nunique().rename("seeds").reset_index().to_string())

## 1. Coverage and the common test set
Only (country, origin) pairs predicted by **every** honest-track run are scored, so all models face exactly the same days.

In [ ]:
honest = allp[allp["track"] == "honest"]
com, common, coverage = cb_common_set(honest, min_coverage=MIN_COVERAGE)
display(coverage.rename("coverage_of_union").to_frame().round(3))
print("Countries in common set:", sorted(common["country"].unique()))

## 2. Leaderboard (honest track)

In [ ]:
m = cb_metrics(com)
board = cb_skill(cb_aggregate_seeds(m, PRIMARY), "Seasonal naive 168h", PRIMARY)
cols = ["notebook", "model", "feature_set", "family", PRIMARY, f"{PRIMARY}_sd", "n_seeds", "MAPE", "WAPE", "RMSE", "MAE",
        "Bias_pct", "Skill_vs_Seasonal naive 168h", "train_seconds", "n_params"]
board = board[[c for c in cols if c in board]]
board.index = np.arange(1, len(board) + 1)
display(board.round(3))
board.to_csv("cb_leaderboard.csv")

## 3. Is the winner significantly better? (Diebold–Mariano, daily loss, HAC lag 7)
Negative DM = the best model has lower error. Seeds are averaged into one daily loss per model.

In [ ]:
com["model_key"] = com["notebook"] + " | " + com["model"] + " | " + com["feature_set"]
def model_daily_loss(key):
    runs = com.loc[com["model_key"] == key, "run"].unique()
    return pd.concat([cb_daily_loss(com, r) for r in runs], axis=1).mean(axis=1)

board["model_key"] = board["notebook"] + " | " + board["model"] + " | " + board["feature_set"]
best = board["model_key"].iloc[0]
best_loss = model_daily_loss(best)
rows = []
for key in board["model_key"].iloc[1:]:
    dm, p = cb_dm_test(best_loss, model_daily_loss(key))
    rows.append({"vs": key, "DM": dm, "p_value": p, "significant_5pct": p < 0.05})
dm_table = pd.DataFrame(rows)
print("Best model:", best)
display(dm_table.round(4))
dm_table.to_csv("cb_dm_vs_best.csv", index=False)

## 4. Best model per family and cost vs accuracy

In [ ]:
fam = board.sort_values(PRIMARY).groupby("family", dropna=False).head(1)
display(fam[["family", "notebook", "model", "feature_set", PRIMARY, "train_seconds"]].round(3))

fig, ax = plt.subplots(figsize=(8, 5))
x = board.dropna(subset=["train_seconds"])
ax.scatter(x["train_seconds"], x[PRIMARY])
for _, r in x.iterrows():
    ax.annotate(f'{r["notebook"]} {r["model"]}', (r["train_seconds"], r[PRIMARY]), fontsize=7)
ax.set_xscale("log"); ax.set_xlabel("training seconds (log)"); ax.set_ylabel(PRIMARY + " (%)")
ax.set_title("Accuracy vs training cost"); plt.show()

## 5. Which feature engineering wins?

**Controlled view:** the reference models (REF-Ridge, REF-MLP) have fixed hyper-parameters and are trained on each notebook's own input representation, so differences come from the features only.

**Uncontrolled view:** the best native model per feature set (model and features both vary).

In [ ]:
ref = board[board["family"] == "reference"].copy()
fe_controlled = ref.pivot_table(index="feature_set", columns="model", values=PRIMARY).sort_values("REF-MLP")
fe_controlled["REF-MLP_sd"] = ref[ref["model"] == "REF-MLP"].set_index("feature_set")[f"{PRIMARY}_sd"]
display(fe_controlled.round(3))

native = board[~board["family"].isin(["reference", "naive"])]
fe_native = native.sort_values(PRIMARY).groupby("feature_set").head(1)[["feature_set", "notebook", "model", PRIMARY]]
display(fe_native.round(3))
fe_controlled.to_csv("cb_fe_controlled.csv"); fe_native.to_csv("cb_fe_native.csv", index=False)

if len(ref):
    fe_best = fe_controlled.index[0]
    a = model_daily_loss(ref.loc[(ref["model"] == "REF-MLP") & (ref["feature_set"] == fe_best), "model_key"].iloc[0])
    rows = []
    for fs in fe_controlled.index[1:]:
        k = ref.loc[(ref["model"] == "REF-MLP") & (ref["feature_set"] == fs), "model_key"]
        if len(k):
            dm, p = cb_dm_test(a, model_daily_loss(k.iloc[0])); rows.append({"vs": fs, "DM": dm, "p_value": p})
    print("Best feature set with REF-MLP:", fe_best); display(pd.DataFrame(rows).round(4))

## 6. Where do models differ? Error by horizon and by country (top 6)

In [ ]:
top_keys = board["model_key"].head(6).tolist()
top_runs = com.loc[com["model_key"].isin(top_keys) & com["seed"].isin([42, 0]), "run"].unique()
ph = cb_per_horizon(com, top_runs)
ph.plot(figsize=(9, 4), marker="o", ms=3); plt.ylabel("MAPE (%)"); plt.xlabel("hours ahead"); plt.title("Error by horizon"); plt.show()

pc = cb_per_country(com, top_runs)
display(pc.round(2).style.background_gradient(axis=1, cmap="RdYlGn_r"))

## 7. Oracle track (uses observed future temperature) — reported separately, never mixed with the honest leaderboard

In [ ]:
oracle = allp[allp["track"] == "oracle"]
if len(oracle):
    o_com, _, _ = cb_common_set(pd.concat([oracle, base]), min_coverage=MIN_COVERAGE)
    display(cb_aggregate_seeds(cb_metrics(o_com), PRIMARY).round(3))
else:
    print("No oracle runs saved.")

## 8. Export

In [ ]:
with pd.ExcelWriter("cb_results.xlsx") as w:
    board.to_excel(w, sheet_name="leaderboard"); dm_table.to_excel(w, sheet_name="dm_vs_best", index=False)
    coverage.rename("coverage").to_frame().to_excel(w, sheet_name="coverage")
    if len(ref): fe_controlled.to_excel(w, sheet_name="fe_controlled")
    fe_native.to_excel(w, sheet_name="fe_native", index=False)
print("Saved cb_leaderboard.csv, cb_dm_vs_best.csv, cb_fe_*.csv, cb_results.xlsx")